In [1]:
import os
os.chdir("../..")

In [12]:
from rdflib import Graph, RDF, RDFS, OWL, Namespace
from neo4j import GraphDatabase
from urllib.parse import urlparse

# PARSING THE GRAPH

In [3]:
g= Graph()
g.parse("./data/raw/marvelcinematicuniverse-marvel/component/target.xml")

<Graph identifier=Naf8da6121eb74a408b67c97687efb65b (<class 'rdflib.graph.Graph'>)>

In [4]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))

In [13]:
def uri_to_name(uri):
    """Convert URI to readable name (last fragment)."""
    parsed = urlparse(str(uri))
    return parsed.fragment or parsed.path.split("/")[-1]

def create_node(tx, label, uri, properties):
    cypher = f"""
    MERGE (n:{label} {{uri: $uri}})
    SET n += $props
    """
    tx.run(cypher, uri=str(uri), props=properties)

def create_relationship(tx, subj, pred, obj):
    rel_type = uri_to_name(pred)
    cypher = f"""
    MATCH (a {{uri: $subj}}), (b {{uri: $obj}})
    MERGE (a)-[r:{rel_type}]->(b)
    """
    tx.run(cypher, subj=str(subj), obj=str(obj))

# INSERT CLASSES AND RELATIONS AMONG THEM

In [ ]:
with driver.session() as session:
    # 1. Create Class nodes
    for s in g.subjects(RDF.type, RDFS.Class):
        props = {
            "name": uri_to_name(s),
            "label": uri_to_name(s),
            "description": next(g.objects(s, RDFS.comment), None),
            "image": None,  # flexible placeholder
        }
        session.execute_write(create_node, "Class", s, props)

    for s in g.subjects(RDF.type, OWL.Class):
        props = {
            "name": uri_to_name(s),
            "label": uri_to_name(s),
            "description": next(g.objects(s, RDFS.comment), None),
            "image": None,
        }
        session.execute_write(create_node, "Class", s, props)

    # 2. Create Instance nodes
    for s, o in g.subject_objects(RDF.type):
        if o in (RDFS.Class, OWL.Class):
            continue  # already handled
        props = {
            "name": uri_to_name(s),
            "label": uri_to_name(s),
            "description": next(g.objects(s, RDFS.comment), None),
            "image": None,
        }
        class_name = uri_to_name(o)  # most specific class
        session.execute_write(create_node, class_name, s, props)

    # 3. Create Relationships
    for s, p, o in g:
        if (p, RDF.type, RDF.Property) in g or isinstance(o, (str, int, float)):
            continue  # handled as attributes
        session.execute_write(create_relationship, s, p, o)

driver.close()